In [2]:
!pip install -q groq pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00


In [7]:
import pandas as pd
import json
import time
from groq import Groq

# --- CONFIGURATION ---
# Replace with your actual Groq API Key
API_KEY = "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"

# Updated to the currently supported model
MODEL = "llama-3.3-70b-versatile"

# Initialize Client
client = Groq(api_key=API_KEY)

# --- LOAD DATA ---
try:
    # Processing first 5 rows for the report
    df = pd.read_csv('yelp.csv', nrows=5, encoding='unicode_escape')
    print("Data loaded successfully.")
except Exception as e:
    print(f"Error loading CSV: {e}")
    df = pd.DataFrame()

# --- ANALYSIS FUNCTION ---
def get_rating_prediction(text, prompt_style):
    """
    Sends the review text to the LLM using the specified prompt style.
    Returns a dictionary with predicted_stars and explanation.
    """

    if prompt_style == "basic":
        system_content = "You are a helpful assistant. Output valid JSON only."
        user_content = f"""
        Analyze the following Yelp review and predict the star rating (1-5).
        Return a JSON object with keys: "predicted_stars" (int) and "explanation" (string).

        Review: "{text}"
        """

    elif prompt_style == "persona":
        system_content = "You are an expert Customer Experience Analyst. Output valid JSON only."
        user_content = f"""
        Based on the sentiment and tone of this review, assign a star rating (1-5).
        Return a JSON object with keys: "predicted_stars" (int) and "explanation" (string).

        Review: "{text}"
        """

    elif prompt_style == "few_shot":
        system_content = "You are a helpful assistant. Output valid JSON only."
        user_content = f"""
        Classify the review rating. Return JSON with keys: "predicted_stars" (int) and "explanation" (string).

        Examples:
        Input: "The food was cold and service was slow." -> Output: 1
        Input: "Absolutely loved the atmosphere and the staff." -> Output: 5

        Current Review: "{text}"
        """
    else:
        return {"predicted_stars": 0, "explanation": "Invalid style"}

    try:
        completion = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_content},
                {"role": "user", "content": user_content}
            ],
            temperature=0,
            response_format={"type": "json_object"}
        )

        response_content = completion.choices[0].message.content
        return json.loads(response_content)

    except Exception as e:
        return {"predicted_stars": 0, "explanation": "API Error"}

# --- MAIN EXECUTION ---
if not df.empty:
    results = []
    print("Starting analysis...")

    for index, row in df.iterrows():
        review_text = row['text']
        actual_stars = row['stars']

        print(f"Processing review ID {index}...")

        for style in ["basic", "persona", "few_shot"]:
            prediction = get_rating_prediction(review_text, style)

            results.append({
                "review_id": index,
                "actual_stars": actual_stars,
                "prompt_style": style,
                "predicted_stars": prediction.get('predicted_stars'),
                "explanation": prediction.get('explanation')
            })

    # Save results to CSV
    output_filename = "task1_results.csv"
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_filename, index=False)

    print(f"Analysis complete. Results saved to {output_filename}.")
    print(results_df.head())

Data loaded successfully.
Starting analysis...
Processing review ID 0...
Processing review ID 1...
Processing review ID 2...
Processing review ID 3...
Processing review ID 4...
Analysis complete. Results saved to task1_results.csv.
   review_id  actual_stars prompt_style  predicted_stars  \
0          0             5        basic                5   
1          0             5      persona                5   
2          0             5     few_shot                5   
3          1             5        basic                5   
4          1             5      persona                5   

                                         explanation  
0  The reviewer uses extremely positive language ...  
1  The reviewer uses extremely positive language ...  
2  The reviewer uses extremely positive language ...  
3  The reviewer had a very positive experience, m...  
4  The reviewer had a very positive experience, p...  
